In [1]:
import os
import openai

OLLAMA_API_URL = os.getenv("OLLAMA_API_URL")

# 클라이언트 초기화
client = openai.OpenAI(
    base_url=OLLAMA_API_URL, # OpenAI API 대신 ollama 사용
    api_key="ollama-key" # 필수로 문자열을 요구하므로 주석 해제 유지
)

# Ollama 서버의 모델 목록 조회
models = client.models.list()

# 사용 가능한 모델의 ID(이름) 출력
print("=== 사용 가능한 모델 목록 ===")
for model in models.data:
    print("-", model.id)


=== 사용 가능한 모델 목록 ===
- gemma4:e4b
- gemma4:e2b
- qwen3.5:4b
- qwen3.5:9b
- nemotron-3-nano:4b
- functiongemma:latest
- translategemma:12b
- translategemma:4b
- nomic-embed-text-v2-moe:latest
- olmo-3:7b
- ministral-3:14b
- ministral-3:8b
- ministral-3:3b
- qwen3-vl:8b
- qwen3-vl:4b
- qwen3-vl:2b
- qwen3-embedding:0.6b
- embeddinggemma:300m
- gemma3n:e2b
- gemma3n:e4b
- linux6200/bge-reranker-v2-m3:latest
- gemma3:4b
- gemma3:1b
- gemma3:12b
- bona/bge-m3-korean:latest
- shieldgemma:latest
- llama-guard3:8b
- llama3.2-vision:11b
- nomic-embed-text:latest
- codegemma:latest


In [2]:
import openai
import json
import requests

# 1. 영화 API 호출 함수
BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"

def get_popular_movies():
    response = requests.get(f"{BASE_URL}/movies")
    return response.json()

def get_movie_details(id):
    response = requests.get(f"{BASE_URL}/movies/{id}")
    return response.json()

def get_movie_credits(id):
    response = requests.get(f"{BASE_URL}/movies/{id}/credits")
    return response.json()

# ⭐️ 추가: 함수 이름(문자열)과 실제 파이썬 함수를 매핑해주는 딕셔너리
available_functions = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits,
}

# 2. 클라이언트 초기화
client = openai.OpenAI(
    base_url=OLLAMA_API_URL,
    api_key="ollama-key" 
)

# 3. 도구(Tools) 정의 (이전과 동일)
tools = [
     {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "현재 인기 있는 영화 목록을 가져옵니다.",
            "parameters": {"type": "object", "properties": {}}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "특정 ID를 가진 영화의 상세 정보를 가져옵니다.",
            "parameters": {
                "type": "object",
                "properties": {"id": {"type": "integer", "description": "영화 고유 ID"}}
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "특정 ID를 가진 영화의 출연진 및 제작진 정보를 가져옵니다.",
            "parameters": {
                "type": "object",
                "properties": {"id": {"type": "integer", "description": "영화 고유 ID"}}
            }
        }
    }
]

test_inputs = [
    "지금 인기 있는 영화가 무엇인지 알려줘",
    "movie ID 550에 해당하는 영화가 무엇인지 알려줘",
    "movie ID 550에 해당하는 영화에 누가 출연하는지 알려줘"
]

print("🎬 Movie Expert Agent + 🛠️ 실제 함수 실행 테스트 시작\n" + "="*50)

for user_input in test_inputs:
    print(f"\n👤 [사용자 입력]: \"{user_input}\"")
    
    response = client.chat.completions.create(
        model="gemma4:e2b",
        messages=[
            {"role": "system", "content": "당신은 사용자의 질문에 알맞은 영화 API 함수를 호출하는 Movie Expert Agent입니다."},
            {"role": "user", "content": user_input}
        ],
        tools=tools,
        tool_choice="auto"
    )
    
    message = response.choices[0].message
    
    if message.tool_calls:
        for tool_call in message.tool_calls:
            # LLM이 선택한 함수 이름과 인자 텍스트
            function_name = tool_call.function.name
            function_args = tool_call.function.arguments
            
            print(f"🤖 [모델의 결정] '{function_name}' 함수를 호출하세요. 인자: {function_args}")
            
            # ⭐️ 실제 함수 실행 로직
            function_to_call = available_functions.get(function_name)
            
            if function_to_call:
                print("   🚀 API에 실제 요청을 보냅니다...")
                try:
                    # JSON 문자열 인자를 파이썬 딕셔너리로 변환
                    args_dict = json.loads(function_args) if function_args else {}
                    
                    # 매핑된 실제 함수 실행 (**args_dict 형태로 키워드 인자 전달)
                    function_result = function_to_call(**args_dict)
                    
                    # API 결과가 너무 길어 화면이 꽉 찰 수 있으니, 결과 텍스트의 앞부분 200자만 출력합니다.
                    # 전체 데이터를 보고 싶으시다면 [str(function_result)[:200]] 이 부분을 [function_result] 로 변경해도 됩니다.
                    print(f"   ✅ [실행 결과 (일부 발췌)]: {str(function_result)[:200]}...")
                except Exception as e:
                     print(f"   ❌ [오류 발생]: {e}")
            else:
                 print(f"   ❌ 알려지지 않은 함수가 호출되었습니다: {function_name}")
    else:
        print(f"🤖 [답변]: {message.content}")

print("\n" + "="*50 + "\n🎬 테스트 종료")


🎬 Movie Expert Agent + 🛠️ 실제 함수 실행 테스트 시작

👤 [사용자 입력]: "지금 인기 있는 영화가 무엇인지 알려줘"
🤖 [모델의 결정] 'get_popular_movies' 함수를 호출하세요. 인자: {}
   🚀 API에 실제 요청을 보냅니다...
   ✅ [실행 결과 (일부 발췌)]: [{'adult': False, 'backdrop_path': 'https://image.tmdb.org/t/p/w1280/1x9e0qWonw634NhIsRdvnneeqvN.jpg', 'genre_ids': [10749, 18], 'id': 1523145, 'original_language': 'ru', 'original_title': 'Твоё сердц...

👤 [사용자 입력]: "movie ID 550에 해당하는 영화가 무엇인지 알려줘"
🤖 [모델의 결정] 'get_movie_details' 함수를 호출하세요. 인자: {"id":550}
   🚀 API에 실제 요청을 보냅니다...
   ✅ [실행 결과 (일부 발췌)]: {'adult': False, 'backdrop_path': 'https://image.tmdb.org/t/p/w1280/c6OLXfKAk5BKeR6broC8pYiCquX.jpg', 'belongs_to_collection': None, 'budget': 63000000, 'genres': [{'id': 18, 'name': 'Drama'}, {'id': ...

👤 [사용자 입력]: "movie ID 550에 해당하는 영화에 누가 출연하는지 알려줘"
🤖 [모델의 결정] 'get_movie_credits' 함수를 호출하세요. 인자: {"id":550}
   🚀 API에 실제 요청을 보냅니다...
   ✅ [실행 결과 (일부 발췌)]: [{'adult': False, 'gender': 2, 'id': 819, 'known_for_department': 'Acting', 'name': 'Edward Norton', 'original